## Task 1 — `promisify` (Deno / TypeScript)

Implement **`promisify`**, which wraps a function `fn` and returns another function whose result is a **Promise** that may fulfill or reject.

### Signature

- **`promisify(fn, hasCallback = true)`**
  - **`fn`**: must be a function; if it is not, **`promisify` must return the string `"invalid"`** (do not throw from there).
  - **`hasCallback`**: boolean, defaults to `true`.

### Behaviour

1. If **`fn` is not a function** → return **`"invalid"`**.
2. Otherwise return a **function** that:
   - accepts **any number of arguments** (variadic calls);
   - returns a **Promise** tied to running `fn`.
3. If **`hasCallback === true`**: assume Node-style **`(error, response) => { ... }`**. The promise should **resolve** with `response` or **reject** with `error`.
4. If **`hasCallback === false`**: treat `fn` as a **plain** function (no trailing callback).


In [ ]:
/**
 * Wraps `fn` so you get a function that returns a Promise.
 *
 * @param fn — function to wrap (if not a function, returns `"invalid"`).
 * @param hasCallback — when true, `fn` follows `(error, response) => void`.
 */
function promisify(
  fn: Function,
  hasCallback = true,
): string | ((...args: unknown[]) => Promise<unknown>) {
  if (typeof fn !== "function") {
    return "invalid";
  }

  return (...args: unknown[]): Promise<unknown> =>
    new Promise((resolve, reject) => {
      if (hasCallback) {
        try {
          fn(...args, (error: unknown, response: unknown) => {
            if (error) reject(error);
            else resolve(response);
          });
        } catch (e) {
          reject(e);
        }
      } else {
        try {
          Promise.resolve(fn(...args)).then(resolve, reject);
        } catch (e) {
          reject(e);
        }
      }
    });
}

// --- Manual examples (uncomment when done) ---
// function readFileCb(path: string, cb: (err: Error | null, data?: string) => void) {
//   queueMicrotask(() => (path === "bad" ? cb(new Error("fail")) : cb(null, "ok")));
// }
// const read = promisify(readFileCb, true);
// console.log(await read("good"));
// try {
//   await read("bad");
// } catch (e) {
//   console.assert((e as Error).message === "fail");
// }
// console.assert(promisify(null as unknown as Function) === "invalid");
